# B. Kiểm Tra Dữ Liệu Thô

In [1]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)

it = pd.read_csv("../data/raw/00-itviec_raw.csv")
tc = pd.read_csv("../data/raw/00-topcv_raw.csv")

print(f"ITViec : {it.shape[0]} hàng  ×  {it.shape[1]} cột")
print(f"TopCV  : {tc.shape[0]} hàng  ×  {tc.shape[1]} cột")

ITViec : 751 hàng  ×  15 cột
TopCV  : 4384 hàng  ×  16 cột


## 1. Tổng quan cấu trúc

In [2]:
# Cột của từng nguồn
it_cols = set(it.columns)
tc_cols = set(tc.columns)
only_tc = tc_cols - it_cols
only_it = it_cols - tc_cols
common  = it_cols & tc_cols

print(f"Cột chung ({len(common)})  : {sorted(common)}")
print(f"Chỉ TopCV ({len(only_tc)}) : {sorted(only_tc)}")
print(f"Chỉ ITViec ({len(only_it)}): {sorted(only_it)}")

Cột chung (15)  : ['company', 'education', 'employment_type', 'experience', 'industry', 'job_description', 'job_title', 'level', 'location', 'preferred_skills', 'required_skills', 'requirement', 'salary', 'specialization', 'url']
Chỉ TopCV (1) : ['brand_recrawled']
Chỉ ITViec (0): []


In [ ]:
# Kiểu dữ liệu + số null của từng cột — ITViec
print("=== ITViec dtypes & null ===")
summary = pd.DataFrame({
    "dtype": it.dtypes,
    "non_null": it.notna().sum(),
    "null": it.isna().sum(),
    "null_%": (it.isna().sum() / len(it) * 100).round(1),
})
print(summary.to_string())

In [ ]:
# Kiểu dữ liệu + số null của từng cột — TopCV
print("=== TopCV dtypes & null ===")
summary_tc = pd.DataFrame({
    "dtype": tc.dtypes,
    "non_null": tc.notna().sum(),
    "null": tc.isna().sum(),
    "null_%": (tc.isna().sum() / len(tc) * 100).round(1),
})
print(summary_tc.to_string())

In [ ]:
# Bảng so sánh tỷ lệ null cạnh nhau — tất cả cột chung
cols_common_list = sorted(common)
null_compare = pd.DataFrame({
    "ITViec_null_%": (it[cols_common_list].isna().sum() / len(it) * 100).round(1),
    "TopCV_null_%":  (tc[cols_common_list].isna().sum() / len(tc) * 100).round(1),
})
null_compare["diff"] = (null_compare["ITViec_null_%"] - null_compare["TopCV_null_%"]).round(1)
print(null_compare.sort_values("ITViec_null_%", ascending=False).to_string())

## 2. Trùng lặp

In [ ]:
# Trùng URL nội bộ từng nguồn
print("Trùng URL nội bộ:")
print(f"  ITViec : {it['url'].duplicated().sum()}")
print(f"  TopCV  : {tc['url'].duplicated().sum()}")

# Trùng URL xuyên nguồn
all_urls = pd.concat([it["url"], tc["url"]], ignore_index=True)
print(f"\nTrùng URL xuyên nguồn (IT url xuất hiện trong TC): "
      f"{it['url'].isin(tc['url']).sum()}")

# Trùng hàng hoàn toàn (tất cả cột chung)
merged = pd.concat([it[cols_common_list], tc[cols_common_list]], ignore_index=True)
print(f"\nTrùng hàng hoàn toàn (tất cả cột chung): {merged.duplicated().sum()}")

# Trùng nội dung mềm: job_title + company + location (case-insensitive)
merged["_key"] = (
    merged["job_title"].str.lower().str.strip().fillna("") + "||" +
    merged["company"].str.lower().str.strip().fillna("")   + "||" +
    merged["location"].str.lower().str.strip().fillna("")
)
n_soft_dup = merged.duplicated(subset="_key").sum()
print(f"Trùng mềm (job_title + company + location): {n_soft_dup}  "
      f"({n_soft_dup / len(merged) * 100:.1f}%)")

# Xem thử 5 cặp trùng mềm
dup_keys = merged[merged.duplicated(subset="_key", keep=False)].sort_values("_key")
print("\nVí dụ 5 nhóm trùng mềm:")
print(dup_keys[["job_title","company","location"]].drop_duplicates().head(5).to_string(index=False))

## 3. Trường `salary`

In [ ]:
# Giá trị thực tế (top 20 phổ biến nhất) — mỗi nguồn
print("=== TopCV — top 20 giá trị salary ===")
print(tc["salary"].value_counts(dropna=False).head(20).to_string())

print("\n=== ITViec — top 20 giá trị salary ===")
print(it["salary"].value_counts(dropna=False).head(20).to_string())

In [ ]:
# Phân loại salary thành 4 nhóm — xem tỷ lệ thực tế
def classify_salary(s):
    if pd.isna(s):
        return "null"
    s2 = str(s).lower().strip()
    if re.search(r"\d\s*[-–]\s*\d", s2):
        return "range"          # dạng khoảng: 15-25 triệu
    if re.search(r"(t[oớ]i|up\s*to|tới)\s*\d", s2):
        return "open_max"       # dạng tới X: Tới 30 triệu
    if re.search(r"(từ|from|thỏa thuận từ)\s*\d", s2):
        return "open_min"       # dạng từ X: Thỏa thuận từ 28 triệu
    if re.search(r"\d", s2):
        return "single_num"     # có số nhưng không rõ kiểu
    return "text_only"          # không có số: Thỏa thuận, Cạnh tranh...

for name, df in [("TopCV", tc), ("ITViec", it)]:
    cats = df["salary"].apply(classify_salary).value_counts(dropna=False)
    pct  = (cats / len(df) * 100).round(1)
    result = pd.DataFrame({"count": cats, "%": pct})
    print(f"=== {name} ===")
    print(result.to_string())
    print()

In [ ]:
# Parse lương → triệu VNĐ/tháng (mid-point) để vẽ phân bố
USD_RATE = 25.0  # 1 USD = 25,000 VNĐ = 0.025 triệu

def parse_salary_vnd(s):
    """Trả về (min_m, max_m) triệu VNĐ/tháng hoặc (None, None)."""
    if pd.isna(s):
        return None, None
    s2 = str(s).lower().strip()
    is_usd = "usd" in s2 or "$" in s2

    nums = [float(n.replace(",", "")) for n in re.findall(r"[\d,]+(?:\.\d+)?", s2)]
    if not nums:
        return None, None

    # USD: ITViec dùng đơn vị đầy đủ (e.g. 1000 USD), không phải triệu
    if is_usd:
        nums = [v / 1000 * USD_RATE for v in nums]  # → triệu VNĐ

    if len(nums) >= 2:
        lo, hi = nums[0], nums[1]
    else:
        lo = hi = nums[0]

    # Loại giá trị vô lý
    if lo > 500 or hi > 500 or lo < 0.5:
        return None, None
    return round(lo, 2), round(hi, 2)

tc["sal_min"], tc["sal_max"] = zip(*tc["salary"].map(parse_salary_vnd))
it["sal_min"], it["sal_max"] = zip(*it["salary"].map(parse_salary_vnd))

tc["sal_mid"] = (tc["sal_min"] + tc["sal_max"]) / 2
it["sal_mid"] = (it["sal_min"] + it["sal_max"]) / 2

print(f"TopCV  — có lương số hóa: {tc['sal_mid'].notna().sum()} / {len(tc)}  "
      f"({tc['sal_mid'].notna().sum()/len(tc)*100:.1f}%)")
print(f"ITViec — có lương số hóa: {it['sal_mid'].notna().sum()} / {len(it)}  "
      f"({it['sal_mid'].notna().sum()/len(it)*100:.1f}%)")

all_mid = pd.concat([tc["sal_mid"], it["sal_mid"]]).dropna()
print(f"\nTổng bản ghi có lương số hóa được: {len(all_mid)}")
print(all_mid.describe(percentiles=[.05, .25, .5, .75, .95]).round(1))

In [ ]:
# Phân bố lương: histogram gốc + log-transform
from scipy import stats as scipy_stats

skew = round(float(scipy_stats.skew(all_mid)), 2)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(all_mid.clip(upper=150), bins=60, color="#4C72B0", edgecolor="white", lw=0.4)
axes[0].axvline(all_mid.median(), color="#e63946", ls="--", lw=1.5,
                label=f"Trung vị = {all_mid.median():.1f}M")
axes[0].axvline(all_mid.mean(),   color="#f4a261", ls="--", lw=1.5,
                label=f"Trung bình = {all_mid.mean():.1f}M")
axes[0].set_xlabel("Lương mid (triệu VNĐ/tháng)  [clip ≤150M]")
axes[0].set_ylabel("Số bản ghi")
axes[0].set_title(f"Phân bố lương thô  (n={len(all_mid):,},  skew={skew})")
axes[0].legend(fontsize=9)

axes[1].hist(np.log1p(all_mid), bins=60, color="#2a9d8f", edgecolor="white", lw=0.4)
axes[1].set_xlabel("log(1 + lương mid)  [triệu VNĐ]")
axes[1].set_ylabel("Số bản ghi")
axes[1].set_title("Sau log-transform")

plt.tight_layout()
plt.savefig("salary_dist.png", dpi=120, bbox_inches="tight")
plt.show()
print(f"Skewness gốc: {skew}  → lệch phải, log-transform cải thiện chuẩn hóa")
print(f"Giá trị cực đoan > 100M: {(all_mid > 100).sum()} bản ghi")
print(f"Giá trị cực đoan > 200M: {(all_mid > 200).sum()} bản ghi")

## 4. Trường `experience`

In [ ]:
# TopCV: dạng categorical rõ ràng
print("=== TopCV — experience (tất cả giá trị unique) ===")
print(tc["experience"].value_counts(dropna=False).to_string())

print("\n=== ITViec — experience ===")
print(f"Null: {it['experience'].isna().sum()}  /  {len(it)}")
print(f"Non-null: {it['experience'].notna().sum()}")
if it["experience"].notna().any():
    print("Mẫu giá trị non-null:")
    print(it["experience"].dropna().head(10).to_string())

## 5. Trường `level`

In [ ]:
print("=== TopCV — level ===")
print(tc["level"].value_counts(dropna=False).to_string())

print("\n=== ITViec — level ===")
print(it["level"].value_counts(dropna=False).to_string())

# Vì ITViec không có level, kiểm tra job_title xem có chứa cấp bậc không
print("\n=== ITViec — job_title (mẫu 15) ===")
print(it["job_title"].dropna().head(15).to_string())

# Thử trích cấp bậc từ job_title ITViec
LEVEL_PATTERN = re.compile(
    r"\b(intern|fresher|junior|mid[\s-]?level|middle|senior|lead|manager|"
    r"principal|director|head|staff|associate)\b", re.I
)
it["level_from_title"] = it["job_title"].str.extract(LEVEL_PATTERN, expand=False)
print("\nCấp bậc trích được từ job_title (ITViec):")
print(it["level_from_title"].value_counts(dropna=False).to_string())

## 6. Trường `location`

In [ ]:
print("=== TopCV — top 20 location ===")
print(tc["location"].value_counts(dropna=False).head(20).to_string())

print("\n=== ITViec — top 20 location ===")
print(it["location"].value_counts(dropna=False).head(20).to_string())

# Kiểm tra format: 1 thành phố hay nhiều?
print("\n=== ITViec — ví dụ location có dấu phân cách ===")
multi = it["location"].dropna()
multi = multi[multi.str.contains(r"[,;|]|\ - ", regex=True)]
print(f"Có dấu phân cách: {len(multi)}")
print(multi.head(10).to_string())

## 7. Trường `required_skills` / `preferred_skills`

In [ ]:
# Kiểm tra format raw của required_skills
for name, df in [("TopCV", tc), ("ITViec", it)]:
    col = df["required_skills"].dropna()
    print(f"=== {name} — required_skills ===")
    print(f"  Non-null: {len(col)} / {len(df)}")
    print("  Mẫu 5 giá trị:")
    for v in col.head(5):
        print(f"    {repr(str(v)[:120])}")
    print()

# Kiểm tra separator
print("Phân tích separator:")
for name, df in [("TopCV", tc), ("ITViec", it)]:
    col = df["required_skills"].dropna().astype(str)
    has_comma   = col.str.contains(",").sum()
    has_newline = col.str.contains(r"\n").sum()
    has_semi    = col.str.contains(";").sum()
    print(f"  {name}: comma={has_comma}  newline={has_newline}  semicolon={has_semi}")

In [ ]:
# Phân bố số lượng skill tags mỗi bản ghi (dùng dấu phẩy làm separator)
def count_skills(s, sep=","):
    if pd.isna(s) or str(s).strip() == "":
        return 0
    return len([x for x in str(s).split(sep) if x.strip()])

for name, df in [("TopCV", tc), ("ITViec", it)]:
    n_skills = df["required_skills"].apply(count_skills)
    print(f"=== {name} — số skill mỗi tin ===")
    print(f"  0 skill (null/rỗng) : {(n_skills == 0).sum()}")
    print(f"  1-5 skill           : {((n_skills >= 1) & (n_skills <= 5)).sum()}")
    print(f"  6-10 skill          : {((n_skills >= 6) & (n_skills <= 10)).sum()}")
    print(f"  >10 skill           : {(n_skills > 10).sum()}")
    print(f"  Median              : {n_skills[n_skills > 0].median():.0f}")
    print(f"  Max                 : {n_skills.max()}")
    print()

## 8. Trường `requirement` (văn bản dài)

In [ ]:
# Trường này là input chính cho annotation — kiểm tra coverage và độ dài
for name, df in [("TopCV", tc), ("ITViec", it)]:
    col = df["requirement"].fillna("").astype(str).str.strip()
    has_text = col.str.len() > 10
    lengths  = col[has_text].str.len()
    print(f"=== {name} — requirement ===")
    print(f"  Có nội dung (>10 ký tự): {has_text.sum()} / {len(df)}")
    print(f"  Độ dài ký tự — min={lengths.min():.0f}  median={lengths.median():.0f}"
          f"  max={lengths.max():.0f}  mean={lengths.mean():.0f}")
    print(f"  Mẫu 1 bản ghi:")
    sample = df.loc[has_text, "requirement"].iloc[0]
    print(f"    {repr(str(sample)[:300])}")
    print()

## 9. Trường `industry` & `employment_type`

In [ ]:
print("=== TopCV — industry (top 15) ===")
print(tc["industry"].value_counts(dropna=False).head(15).to_string())

print("\n=== ITViec — industry (top 15) ===")
print(it["industry"].value_counts(dropna=False).head(15).to_string())

print("\n=== TopCV — employment_type ===")
print(tc["employment_type"].value_counts(dropna=False).to_string())

print("\n=== ITViec — employment_type ===")
print(it["employment_type"].value_counts(dropna=False).to_string())

## 10. Tóm tắt số liệu thô

In [ ]:
n_tc, n_it = len(tc), len(it)
n_total = n_tc + n_it

tc_sal_num = tc["sal_mid"].notna().sum()
it_sal_num = it["sal_mid"].notna().sum()

print("=" * 60)
print("TÓM TẮT SỐ LIỆU THÔ")
print("=" * 60)
print(f"\nKích thước tập dữ liệu")
print(f"  TopCV  : {n_tc:,} bản ghi  ×  {tc.shape[1]} cột")
print(f"  ITViec : {n_it:,} bản ghi  ×  {it.shape[1]} cột")
print(f"  Tổng   : {n_total:,} bản ghi")
print(f"  Cột chung: {len(common)}  |  Chỉ TC: {len(only_tc)}  |  Chỉ IT: {len(only_it)}")

print(f"\nTỷ lệ lương số hóa được (có giá trị số)")
print(f"  TopCV  : {tc_sal_num:,} / {n_tc:,}  ({tc_sal_num/n_tc*100:.1f}%)")
print(f"  ITViec : {it_sal_num:,} / {n_it:,}  ({it_sal_num/n_it*100:.1f}%)")
print(f"  Tổng   : {tc_sal_num+it_sal_num:,} / {n_total:,}  "
      f"({(tc_sal_num+it_sal_num)/n_total*100:.1f}%)")

print(f"\nPhân bố lương (n={len(all_mid):,} bản ghi số hóa được, triệu VNĐ/tháng)")
print(f"  Trung vị   : {all_mid.median():.1f}")
print(f"  Trung bình : {all_mid.mean():.1f}")
print(f"  P5 / P95   : {np.percentile(all_mid,5):.1f}  /  {np.percentile(all_mid,95):.1f}")
print(f"  Skewness   : {scipy_stats.skew(all_mid):.2f}")

print(f"\nTrường thiếu đáng chú ý")
for col in ["level", "experience", "education", "specialization"]:
    tc_m = tc[col].isna().sum() / n_tc * 100
    it_m = it[col].isna().sum() / n_it * 100
    print(f"  {col:20s}  TC={tc_m:5.1f}%   IT={it_m:5.1f}%")

print(f"\nTrùng lặp")
print(f"  Trùng URL nội bộ        : TC={tc['url'].duplicated().sum()}  IT={it['url'].duplicated().sum()}")
print(f"  Trùng URL xuyên nguồn   : {it['url'].isin(tc['url']).sum()}")
print(f"  Trùng nội dung mềm      : {n_soft_dup}  ({n_soft_dup/n_total*100:.1f}%)")

In [1]:
import pandas as pd
import numpy as np
import re
import warnings
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy import stats as scipy_stats

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.1f}".format)

# ── Load dữ liệu thô ──────────────────────────────────────────────────────────
it = pd.read_csv("../data/raw/00-itviec_raw.csv")
tc = pd.read_csv("../data/raw/00-topcv_raw.csv")

it["source"] = "itviec"
tc["source"] = "topcv"

print(f"ITViec : {it.shape[0]:,} bản ghi × {it.shape[1]-1} trường")
print(f"TopCV  : {tc.shape[0]:,} bản ghi × {tc.shape[1]-1} trường")
print(f"Tổng   : {it.shape[0] + tc.shape[0]:,} bản ghi")

ITViec : 751 bản ghi × 15 trường
TopCV  : 4,384 bản ghi × 16 trường
Tổng   : 5,135 bản ghi
